## Restricted Boltzmann Machine

In this example an RBM model is constructed using PyTorch and then training using the MNIST digits dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import trange
import math

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

Define an RBM class using PyTorch

In [ ]:
class BernoulliRBM(nn.Module):
    def __init__(self, visible_units, hidden_units, k=1, learning_rate=1e-3, device=device):
        super(BernoulliRBM, self).__init__()
        self.visible_units = visible_units
        self.hidden_units = hidden_units
        self.k = k
        self.learning_rate = learning_rate
        self.device = device
        
        self.W = nn.Parameter(torch.randn(visible_units, hidden_units, device=self.device) * 0.01, requires_grad=True)
        self.b_v = nn.Parameter(torch.zeros(visible_units, device=self.device), requires_grad=True)
        self.b_h = nn.Parameter(torch.zeros(hidden_units, device=self.device), requires_grad=True)

    def _sample(self, probs):
        return F.relu(torch.sign(probs - torch.rand_like(probs)))

    def contrastive_divergence(self, input_data):
        v_0 = input_data.to(self.device)
        h_0_probs = torch.sigmoid(torch.matmul(v_0, self.W) + self.b_h)
        h_0 = self._sample(h_0_probs)

        for _ in range(self.k):
            v_k_probs = torch.sigmoid(torch.matmul(h_0, self.W.t()) + self.b_v)
            v_k = self._sample(v_k_probs)
            h_k_probs = torch.sigmoid(torch.matmul(v_k, self.W) + self.b_h)
            h_k = self._sample(h_k_probs)

        return v_0, h_0_probs, v_k, h_k_probs

    def train_step(self, input_data):
        v_0, h_0_probs, v_k, h_k_probs = self.contrastive_divergence(input_data)
        positive_grad = torch.matmul(v_0.t(), h_0_probs)
        negative_grad = torch.matmul(v_k.t(), h_k_probs)

        self.W.data += self.learning_rate * (positive_grad - negative_grad) / input_data.shape[0]
        self.b_v.data += self.learning_rate * torch.mean(v_0 - v_k, dim=0)
        self.b_h.data += self.learning_rate * torch.mean(h_0_probs - h_k_probs, dim=0)


    def train(self, train_tensor, batch_size=100, epochs=10):
        num_samples = train_tensor.shape[0]
        num_batches = math.ceil(num_samples / batch_size)

        tqdm_epoch = trange(epochs)
        for epoch in tqdm_epoch:
            # Shuffle the data at the beginning of each epoch
            indices = torch.randperm(num_samples)
            train_tensor_shuffled = train_tensor[indices]

            for batch_idx in range(num_batches):
                start_idx = batch_idx * batch_size
                end_idx = min(start_idx + batch_size, num_samples)
                batch = train_tensor_shuffled[start_idx:end_idx]
            self.train_step(batch)
            tqdm_epoch.set_description(f"Epoch: {epoch + 1}/{epochs}")

    def sample(self, num_samples=1, num_gibbs_steps=1000):
        samples = torch.zeros((num_samples, self.visible_units), device=self.device)

        for i in range(num_samples):
            v_k = torch.bernoulli(torch.rand(self.visible_units)).to(self.device)
            for _ in range(num_gibbs_steps):
                h_k_probs = torch.sigmoid(torch.matmul(v_k, self.W) + self.b_h)
                h_k = self._sample(h_k_probs)
                v_k_probs = torch.sigmoid(torch.matmul(h_k, self.W.t()) + self.b_v)
                v_k = self._sample(v_k_probs)

            samples[i] = v_k

        return samples.detach().cpu().numpy()
    
    def predict(self, mini_batch, steps):
        v_k = mini_batch.to(self.device)
        for k in range(steps):
            h_k_probs = torch.sigmoid(torch.matmul(v_k, self.W) + self.b_h)
            h_k = self._sample(h_k_probs)
            v_k_probs = torch.sigmoid(torch.matmul(h_k, self.W.t()) + self.b_v)
            v_k = self._sample(v_k_probs)
        return v_k.detach().cpu().numpy()

### Load MNIST

Load MNIST numbers dataset and convert to black and white representation

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
train_set = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=True, transform=transform)
test_set = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=False, transform=transform)

In [ ]:
train_x = train_set.data.reshape((60000, 28*28))
train_x = train_x.float() / 255
test_x = test_set.data.reshape((10000, 28 * 28))
test_x = test_x.float() / 255

In [ ]:
train_x = torch.where(train_x > 0.5, 1.0, 0.0).to(device)

In [ ]:
test_x = torch.where(train_x > 0.5, 1.0, 0.0).to(device)

### Train the RBM

In [ ]:
nv = 28*28
nh = 128

In [ ]:
learning_rate = 0.01
batch_size = 100
no_epochs = 100000
cd_steps = 10

In [ ]:
rbm = BernoulliRBM(visible_units=nv, hidden_units=nh, k=cd_steps, learning_rate=learning_rate, device=device)

In [ ]:
rbm.train(train_x, batch_size=batch_size, epochs=no_epochs)

Reconstruct some test images

In [ ]:
reconstructed = rbm.predict(test_x[0:100, :], 1)

In [ ]:
def gen_mnist_image(X):
    return np.rollaxis(np.rollaxis(X[0:200].reshape(20, -1, 28, 28), 0, 2), 1, 3).reshape(-1, 20 * 28)

Plot the reconstructed images

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline   

plt.figure(figsize=(10,20))
plt.imshow(gen_mnist_image(reconstructed[:40]), cmap='gray')
plt.axis("off")
recon_filename = 'rbm_mnistrecon.png'
plt.savefig(recon_filename, dpi=300, bbox_inches='tight')

Plot the original test images

In [ ]:
plt.figure(figsize=(10,20))
plt.imshow(gen_mnist_image(test_x[:40].cpu().numpy()), cmap='gray')
plt.axis("off")
test_filename = 'rbm_mnisttest.png'
plt.savefig(test_filename, dpi=300, bbox_inches='tight')

Generate the some input date using the Bernoulli distribution with probability 0.5.

In [ ]:
rand_x = torch.distributions.Bernoulli(probs=0.5).sample((40, 784)).float()

Run the model until "thermal" equilibrium has been reached

In [ ]:
rand_mnist = rbm.predict(rand_x, 10000)

In [ ]:
rand_mnist.shape

In [ ]:
plt.figure(figsize=(10,20))
plt.imshow(gen_mnist_image(rand_mnist), cmap='gray')
plt.axis("off")
rand_filename = 'rbm_mnistrand.png'
plt.savefig(rand_filename, dpi=300, bbox_inches='tight')

In [ ]:
torch.save(rbm.state_dict(), 'rbm.pth')